In [12]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 
matplotlib.use('QtAgg') 

In [16]:
## Setup
subject = 'RL13MT'
block = '04' #nap number

to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']

inter_trigger_length = 30

raw_path= "/Users/zeynepozkaya/Desktop/Consciousness_Research/Python_Scripts/full_EEG_dataset"

In [14]:
#Functions
def plot_figure():
    global epoch, frq, raw, inter_trigger_length, subject_name
    fig, ax = plt.subplots(3, 1, figsize=(12, 4), sharex=True)
    title = fig.suptitle(f"Epoch {epoch} - Subject {subject_name}")

    # Plot Corr
    ax[0].plot(range(int(epoch*30*frq-1),int(epoch*30*frq+inter_trigger_length*frq)), raw.get_data(picks=['Corr'],start=int(epoch*30*frq-1),stop=int(epoch*30*frq+inter_trigger_length*frq))[0], label="Corr", color="purple",linewidth=0.5)
    # range(int(epoch*30*frq-1) takes start sample of the index where we are in a 30 second window, inter_trigger_length (how many samples forward to include)
    ax[0].set_ylim(-100, 100)
    ax[0].set_ylabel("Corr")
    ax[0].set_xlabel("samples")

    # Plot Zygo
    ax[1].plot(range(int(epoch*30*frq-1),int(epoch*30*frq+inter_trigger_length*frq)), raw.get_data(picks=['Zygo'],start=int(epoch*30*frq-1),stop=int(epoch*30*frq+inter_trigger_length*frq))[0], label="Zygo", color="green",linewidth=0.5)
    ax[1].set_ylim(-100, 100)
    ax[1].set_ylabel("Zygo")
    ax[1].set_xlabel("samples")

    # Plot Chin
    ax[2].plot(range(int(epoch*30*frq-1),int(epoch*30*frq+inter_trigger_length*frq)), raw.get_data(picks=['Menton'],start=int(epoch*30*frq-1),stop=int(epoch*30*frq+inter_trigger_length*frq))[0], label="Chin", color="orange",linewidth=0.5)
    ax[2].set_ylim(-100, 100)
    ax[2].set_ylabel("Chin")
    ax[2].set_xlabel("samples")


    plt.tight_layout()
    plt.show()
    # fig.savefig(f"{subject_name}_epoch_{epoch}.png")
    #plt.close()

In [17]:
# Preprocess
subject_name = subject + block
file= op.join(raw_path,'{}.edf'.format(subject_name))
raw =  mne.io.read_raw_edf(file,preload=True)

if 'Fp1/F3' in raw.info['ch_names']:
    mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
    
if '36' in raw.info['ch_names']:
    mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

if 'E1' in raw.info['ch_names']:
    mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})

raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

for ch in raw.info['ch_names']:
    if ch not in to_keep:
        raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

raw= raw.resample(sfreq=250)

filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

frq=raw.info['sfreq']

Extracting EDF parameters from /Users/zeynepozkaya/Desktop/Consciousness_Research/Python_Scripts/full_EEG_dataset/RL13MT04.edf...


Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 4699199  =      0.000 ...  2294.531 secs...


/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_59088/7178996.py:15: RuntimeWarning: The unit for channel(s) Trigger has changed from NA to V.
  raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})


Finding events on: Trigger
5474 events found on stim channel Trigger
Event IDs: [  4  24  34  61  62  63 201 202 231 232 233 234 241 242 243 244 255]
Finding events on: Trigger
3519 events found on stim channel Trigger
Event IDs: [  4  24  34  61  62  63 201 202 231 232 233 234 241 242 243 244 255]
Filtering a subset of channels. The highpass and lowpass values in the measurement info will not be updated.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 1e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 100.00 Hz
- Upper transition bandwidth: 25.00 Hz (-6 dB cutoff frequency: 112.50 Hz)
- Filter length: 331 samples (1.324 s)

Filtering raw data in 1

/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_59088/7178996.py:21: RuntimeWarning: Resampling of the stim channels caused event information to become unreliable. Consider finding events on the original data and passing the event matrix as a parameter.
  raw= raw.resample(sfreq=250)


Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 1651 samples (6.604 s)



In [ ]:
# Plot figure
epoch = 60 # Not zero and must be < 60 , otherwise you need a condition if epoch-==0 in the function
plot_figure()


### If you want an interactive plot to navigate - only sound trials

In [5]:
# find_events gives an array of events with each element as a sampel number with a trigger ID 
events_all= mne.find_events(raw)
events_all
events= mne.pick_events(events_all,include=[201,202])
events

Finding events on: Trigger
1347 events found on stim channel Trigger
Event IDs: [  1  10  11  12  13  15  16  21  31  61  62  63 105 111 112 201 202 231
 232 233 234 241 242 243 244 255]


array([[334281,      0,    202],
       [337044,      0,    202],
       [339623,      0,    201],
       [342379,      0,    201],
       [345161,      0,    201],
       [347740,      0,    202],
       [364263,      0,    202],
       [367045,      0,    202],
       [369481,      0,    201],
       [372113,      0,    202],
       [374621,      0,    201],
       [377377,      0,    201],
       [394265,      0,    201],
       [396766,      0,    202],
       [399189,      0,    201],
       [401880,      0,    201],
       [404669,      0,    202],
       [406987,      0,    202],
       [424267,      0,    201],
       [426566,      0,    202],
       [429335,      0,    201],
       [431778,      0,    201],
       [434534,      0,    202],
       [437178,      0,    202],
       [454263,      0,    202],
       [456764,      0,    202],
       [459108,      0,    201],
       [461420,      0,    201],
       [464020,      0,    202],
       [466560,      0,    201],
       [48

In [ ]:
print(np.shape(events))

In [6]:
# Check events and create a dataframe with timings
events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)
events[0][0] # gives a single sample (where first event is )

df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
df_triggers.drop(columns=['dunno'],inplace=True)
df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']
df_triggers

Finding events on: Trigger
1347 events found on stim channel Trigger
Event IDs: [  1  10  11  12  13  15  16  21  31  61  62  63 105 111 112 201 202 231
 232 233 234 241 242 243 244 255]


,Time(Sample),Trigger,Time(s)
0,334281,202,1337.124
1,337044,202,1348.176
2,339623,201,1358.492
3,342379,201,1369.516
4,345161,201,1380.644
5,347740,202,1390.960
6,364263,202,1457.052
7,367045,202,1468.180
8,369481,201,1477.924
9,372113,202,1488.452


In [19]:
#GUI
frq=raw.info['sfreq']
current_index=0
inter_trigger_length=10

def plot_figure(t):
    global df_triggers
    fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    title = fig.suptitle(f"Epoch {t + 1}")

    # truncates first few minutes because starts at when first stimulus event occured (after 1 min)
                 
    # Plot Corr
    ax[0].plot(np.arange(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))/frq, raw.get_data(picks=['Corr'],start=int(df_triggers['Time(Sample)'][t]-1),stop=int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))[0], label="Corr", color="blue")
    ax[0].set_ylim(-100, 100)
    ax[0].set_ylabel("Corr")
   # ax[0].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))

    # Plot Zygo
    ax[1].plot(np.arange(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))/frq, raw.get_data(picks=['Zygo'],start=int(df_triggers['Time(Sample)'][t]-1),stop=int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))[0], label="Zygo", color="black")
    ax[1].set_ylim(-100, 100)
    ax[1].set_ylabel("Zygo")
   # ax[1].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))


    # Plot the trigger channel
    ax[2].plot(np.arange(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))/frq, raw.get_data(picks=['Trigger'],start=int(df_triggers['Time(Sample)'][t]-1),stop=int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))[0], label="Trigger Channel", color="orange")
    ax[2].set_xlabel("Time (s)")
    ax[2].set_ylabel("Trigger")
    ax[2].set_ylim(0, 250)
    #ax[2].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))


    # Connect mouse click and key press events
    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.tight_layout()
    plt.show()


# Keyboard press event handler
def on_key(event):
    global current_index, fig, df_triggers
    key = event.key
        
    if event.key == 'right':  # Move to next figure
        current_index = (current_index + 1) % len(df_triggers)  # Loop to the start
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure
    elif event.key == 'left':  # Move to previous figure
        current_index = (current_index - 1) % len(df_triggers)  # Loop to the end
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'escape':  
        print("Quitting the plot!")
        plt.close(fig)  
        

# Plot the first figure
plot_figure(current_index)


KeyboardInterrupt: 

### If you want to go through all epochs by 30s

In [ ]:
#GUI
frq=raw.info['sfreq']
current_index=0
inter_trigger_length=30

def plot_figure(t):
    global df_triggers
    fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    title = fig.suptitle(f"Epoch {int(t/(frq*inter_trigger_length))+1}")
                 
    # Plot Corr
    ax[0].plot(np.arange(t,int(t+inter_trigger_length*frq))/frq, raw.get_data(picks=['Corr'],start=t,stop=int(t+inter_trigger_length*frq))[0], label="Corr", color="blue")
    ax[0].set_ylim(-100, 100)
    ax[0].set_ylabel("Corr")

    # Plot Zygo
    ax[1].plot(np.arange(t,int(t+inter_trigger_length*frq))/frq, raw.get_data(picks=['Zygo'],start=t,stop=int(t+inter_trigger_length*frq))[0], label="Zygo", color="black")
    ax[1].set_ylim(-100, 100)
    ax[1].set_ylabel("Zygo")

    # Plot the trigger channel
    ax[2].plot(np.arange(t,int(t+inter_trigger_length*frq))/frq, raw.get_data(picks=['Trigger'],start=t,stop=int(t+inter_trigger_length*frq))[0], label="Trigger", color="orange")
    ax[2].set_xlabel("Time (s)")
    ax[2].set_ylabel("Trigger")
    ax[2].set_ylim(0, 250)

    # Connect mouse click and key press events
    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.tight_layout()
    plt.show()


# Keyboard press event handler
def on_key(event):
    global current_index, fig, df_triggers
    key = event.key
        
    if event.key == 'right':  # Move to next figure
        current_index = int(current_index + 30*frq) % len(raw) # Loop to the start
        print(current_index)
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure
    elif event.key == 'left':  # Move to previous figure
        current_index = int(current_index - 30*frq) %  len(raw)# Loop to the end
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'escape':  
        print("Quitting the plot!")
        plt.close(fig)  
        

# Plot the first figure
plot_figure(current_index)


In [ ]:
## To epoch the data
# events is matrix of sample at which a word or pseudo word was presented 
# tmin and #tmax is start and end time of epoch in second s
epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                reject=None, preload=True, on_missing='warn') # epoch the data


epochs[0].get_data(picks=['Zygo']) # Get data from one epoch and one channel



#epochs.metadata = #put the filtered csv file for this participant

Test Space for Data Epoching Function 

In [ ]:
# read in csv file containing meta-data
cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc)
expected_muscle = ["Corr", "Zygo"]

epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                reject=None, preload=True, on_missing='warn')

epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                             & (trial_info["Trigger"].isin([201.0, 202.0]))
                             & (trial_info["Nap_ID"] == int(block))]


# adding metadata column for true activation
# add another column for neither muscle being activated  
true_activations = [] 
for i in range(len(epochs.metadata)):
    true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
    if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
        if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
            true_activations.append("None")
        else:
            true_activations.append(expected_muscle[true_ind-1])
    else:
        true_activations.append(expected_muscle[true_ind])


epochs.metadata["True_activation"] = true_activations

    


In [ ]:
# gets features in a sliding window of size window samples with a step size of a certain number of samples 
def get_features(epoch, window,step):
     global frq

     # pad epoch to preserve sample number 
     pad_left  = window // 2
     pad_right = window - 1 - pad_left   
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window 
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]


     var = np.var(epoch_sw, axis=-1) # calculate variance over window 
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  

     # frequency features 
     X = np.fft.rfft(epoch_sw, axis=1)
     PSD = (1/(frq*window)) * np.abs(X)**2
     cumulative = np.cumsum(PSD, axis=1)
     total_power = cumulative[:, -1]
     half_power = total_power / 2

     indices = (cumulative >= half_power[:, None]).argmax(axis=1)
     frequencies = np.fft.rfftfreq(window, 1/frq)

     fmd = frequencies[indices]

     ''' 
     print(np.shape(wl))
     print(np.shape(var))
     print(np.shape(rms))
     print(np.shape(fmd))
     '''
          

     return var, rms, wl, fmd


In [ ]:
#GUI
frq=raw.info['sfreq']
current_index=0
inter_trigger_length=10
window = 50
step = 1

def plot_figure(t):
    global df_triggers
    fig, ax = plt.subplots(4, 1, figsize=(12, 8), sharex=False)
    title = fig.suptitle(f"Epoch {t + 1}, Muscle Activated: {epochs[t].metadata['True_activation'].iloc[0]}, Corr:{epochs[t].metadata['Nb_Corr'].iloc[0]}, Zygo:{epochs[t].metadata['Nb_Zygo'].iloc[0]}")

    epoch_zygo = np.squeeze(epochs[t].get_data(picks=['Zygo']))
    epoch_corr = np.squeeze(epochs[t].get_data(picks=['Corr']))

    epoch_zygo_var, epoch_zygo_rms, epoch_zygo_wl, epoch_zygo_fmd = get_features(epoch_zygo,window,step)
    epoch_corr_var, epoch_corr_rms, epoch_corr_wl, epoch_corr_fmd = get_features(epoch_corr ,window,step)

    # for plotting freq spectrum 
    T = 1.0 / 800.0
    xf = fftfreq(len(epoch_zygo), T)[:len(epoch_zygo)//2]

    # finding peaks 
    prom_z = np.median(np.abs(epoch_zygo_var - np.median(epoch_zygo_var)))
    prom_c = np.median(np.abs(epoch_corr_var - np.median(epoch_corr_var)))

    peaks_zygo, _ = find_peaks(epoch_zygo_var,height=np.mean(epoch_zygo_var), width=30)
    peaks_corr, _ = find_peaks(epoch_corr_var,height=np.mean(epoch_corr_var), width=30)
   

    # truncates first few minutes because starts at when first stimulus event occured (after 1 min)?
                 
    # Plot Corr
    #ax[0].plot(xf,abs(fft(epoch_corr)[0:len(epoch_zygo)//2]), label="Corr", color="blue")
    ax[0].plot(epoch_corr, color="blue") #, label="Corr", 
    ax[0].plot(epoch_corr_rms, label="rms", color="orange")
    ax[0].plot(epoch_corr_var, label="var", color="red")
    ax[0].plot(epoch_corr_wl, label="wl", color="green")
    ax[0].plot(peaks_corr, epoch_corr_var[peaks_corr], "x")
    ax[0].plot(epoch_corr_fmd, label="fmd", color="purple")
    ax[0].set_ylim(-50, 2000)
    ax[0].legend()
    ax[0].set_ylabel("Corr")
    ax[0].set_xlabel("Samples")
   # ax[0].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))

    # Plot Zygo
   # ax[1].plot(xf,abs(fft(epoch_zygo)[0:len(epoch_zygo)//2]), label="Zygo", color="black")
    ax[1].plot(epoch_zygo,label="Zygo", color="black")  
    ax[1].plot(epoch_zygo_rms, label="rms", color="orange")
    ax[1].plot(epoch_zygo_var, label="var", color="red")
    ax[1].plot(epoch_zygo_wl, label="wl", color="green")
    ax[1].plot(peaks_zygo, epoch_zygo_var[peaks_zygo], "x")
    ax[1].set_ylim(-50, 2000)
    #ax[1].plot(epoch_zygo_fmd, label="fmd", color="purple")
   # ax[1].set_ylim(-100,4000)
    ax[1].set_ylabel("Zygo")
    ax[1].set_xlabel("Samples")


    # Plot the trigger channel
    ax[2].plot(np.arange(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))/frq, raw.get_data(picks=['Trigger'],start=int(df_triggers['Time(Sample)'][t]-1),stop=int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))[0], label="Trigger Channel", color="orange")
    ax[2].set_xlabel("Time (s)")
    ax[2].set_ylabel("Trigger")
    ax[2].set_ylim(200, 203)
    #ax[2].set_xlim(int(df_tri

    ax[3].plot(xf,abs(fft(epoch_zygo)[0:len(epoch_zygo)//2]), label="Zygo", color="blue")
    ax[3].plot(xf,abs(fft(epoch_corr)[0:len(epoch_zygo)//2]), label="Corr", color="black")
    ax[3].set_title("Frequency Spectrum") 
    ax[3].set_ylabel("Power")
    ax[3].set_xlabel("Frequency [Hz]")
    ax[3].legend()
   # ax[3].set_xlim(0,len(epoch_zygo)//2)
    
    # Connect mouse click and key press events
    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.tight_layout()
    plt.show()


# Keyboard press event handler
def on_key(event):
    global current_index, fig, df_triggers
    key = event.key
        
    if event.key == 'right':  # Move to next figure
        current_index = (current_index + 1) % len(df_triggers)  # Loop to the start
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure
    elif event.key == 'left':  # Move to previous figure
        current_index = (current_index - 1) % len(df_triggers)  # Loop to the end
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'escape':  
        print("Quitting the plot!")
        plt.close(fig)  
        

# Plot the first figure
plot_figure(current_index)

Test Space for Classification 

In [ ]:
# compares power of different measures of muscle activations 
# and returns which muslce was activated based on which has highest power 
def compare_power(zygo,corr):
    zygo_power = np.mean(zygo**2)
    corr_power = np.mean(corr**2)

    _, zygo_power = welch(zygo, frq, nperseg=1024)
    _, corr_power = welch(corr, frq, nperseg=1024)

    if (np.abs(zygo_power - corr_power) < 30): # hard code threshold for activations to be same (did a relative comparison because other subjects may have different thresholds)
        return "None"
    elif ((zygo_power - corr_power) > 0):  
        return "Zygo"
    else:
        return "Corr"

# determines activation based on power and features -- currently comparing between three and classifying if two match -- only issue is if there are two or so contractions it is not classified as none here 
def determine_activation(zygo,corr,zygo_var,corr_var,zygo_wl,corr_wl):
    # compare power of signal and features 
    sig_power = compare_power(zygo,corr)
    var_power = compare_power(zygo_var,corr_var)
    wl_power = compare_power(zygo_wl,corr_wl)
  
    if (sig_power == 'None'):
        muscle_activated = "None"
    elif (sig_power == var_power == wl_power):
        muscle_activated = sig_power
    elif sig_power == var_power or sig_power == wl_power:
        muscle_activated = sig_power
    elif var_power == wl_power:
        muscle_activated = var_power
    else:
        muscle_activated = "conflict"
    
    return muscle_activated 


In [ ]:
# initialize df to store results of peak finding 
activation_results = pd.DataFrame(
    index=range(len(epochs)),
    columns=[
        "Triggers_Order_Nap",
        "Muscle_Activated_Data",
        "True_Muscle_Activated",
        "Num_Contractions_Zygo_Data",
        "Num_Contractions_Zygo",
        "Num_Contractions_Corr_Data",
        "Num_Contractions_Corr",
        "Contractions_Zygo_Ind",
        "Contractions_Corr_Ind",
        "Activation_Match",
        "Match_Zygo",
        "Match_Corr",
        "Total_Match", # if everything matches (muscle activated and correct contractions)
        "Classification_Match"
    ]
)

muscle_activation_match = [] # for debugging purposes 
# loop through each of the epochs 
for t in range(len(epochs)): 

    # extract epoch  
    epoch_zygo = np.squeeze(epochs[t].get_data(picks=['Zygo']))
    epoch_corr = np.squeeze(epochs[t].get_data(picks=['Corr']))

    # calculate some features 
    epoch_zygo_var, epoch_zygo_rms, epoch_zygo_wl, epoch_zygo_fmd = get_features(epoch_zygo,window,step)
    epoch_corr_var, epoch_corr_rms, epoch_corr_wl, epoch_corr_fmd = get_features(epoch_corr  ,window,step)

    muscle_activated =  determine_activation(epoch_zygo,epoch_corr,epoch_zygo_var,epoch_corr_var,epoch_zygo_wl,epoch_corr_wl)
    activation_match = epochs[t].metadata['True_activation'].iloc[0]==muscle_activated

    muscle_activation_match.append(activation_match)

    # finding peaks relative to the muscle activated 
    prom_z = np.median(np.abs(epoch_zygo_var - np.median(epoch_zygo_var)))
    prom_c = np.median(np.abs(epoch_corr_var - np.median(epoch_corr_var)))

    if  (muscle_activated == "None"):
        peaks_zygo, _ = find_peaks(epoch_zygo_var,height=np.mean(epoch_zygo_var), width=30)
        peaks_corr, _ = find_peaks(epoch_corr_var,height=np.mean(epoch_corr_var), width=30)
    else:
        if (muscle_activated == "Zygo"):
            height = np.mean(epoch_zygo_var) 
        elif (muscle_activated == "Corr"):
            height = np.mean(epoch_corr_var)
        peaks_zygo, _ = find_peaks(epoch_zygo_var,height=height, width=30)
        peaks_corr, _ = find_peaks(epoch_corr_var,height=height, width=30)
    

    num_contractions_zygo_data = len(peaks_zygo) # number of contractions from data 
    num_contractions_corr_data = len(peaks_corr) # number of contractions from data 

    match_zygo = num_contractions_zygo_data == epochs.metadata.iloc[t]["Nb_Zygo"]
    match_corr = num_contractions_corr_data == epochs.metadata.iloc[t]["Nb_Corr"]

    # check match 
    if (activation_match):
        if (muscle_activated == "Zygo"):
            match_total = match_zygo
        elif (muscle_activated == "Corr"):
            match_total = match_corr
        elif (muscle_activated == "None"):
            match_total = activation_match
        else:
            match_total = False
    else:
        match_total = False

    # match based on classification 
    if (activation_match):
        if (muscle_activated == "None"):
            classification_match = activation_match
        else:
            if (num_contractions_zygo_data >= 2):
                classify = "Zygo"
            elif (num_contractions_corr_data >= 2):
                classify = "Corr"
            else:
                classify = "None"
            classification_match = (classify == epochs[t].metadata['True_activation'].iloc[0])
             
    else:
        classification_match = False

    

    # fill dataframe 
    activation_results.loc[t] = [
        t + 1,
        muscle_activated,
        epochs[t].metadata['True_activation'].iloc[0],
        num_contractions_zygo_data,
        epochs.metadata.iloc[t]["Nb_Zygo"],
        num_contractions_corr_data,
        epochs.metadata.iloc[t]["Nb_Corr"],
        peaks_zygo,
        peaks_corr,
        int(activation_match),
        int(match_zygo), 
        int(match_corr),
        int(match_total), 
        int(classification_match)
    ]

      
    

conflict_indices = [i+1 for i, val in enumerate(muscle_activation_match) if not val] # indices where the muscle activation determined from activity does not match data 

In [ ]:
activation_results.to_excel("activation_results_RL04VF04.xlsx", index=False)

In [ ]:
print(activation_results[["Classification_Match"]])
conflict_indices = [i+1 for i, val in enumerate(activation_results["Classification_Match"].tolist()) if not val] 
print(conflict_indices)

In [ ]:
print(activation_results[["Muscle_Activated_Data", "Num_Contractions_Corr_Data", "Num_Contractions_Corr"]])

In [ ]:
# display data frame 
print(activation_results[["Muscle_Activated_Data", "Num_Contractions_Zygo_Data", "Num_Contractions_Zygo", "Num_Contractions_Corr_Data", "Num_Contractions_Corr"]])